# Erasmus KA1 Mobility Analysis – Data Preparation

## Project Overview

 In particular, we will only be looking at the data pertaining university students.
The relevant datasets can be found at the dedicated [European Commission page](https://erasmus-plus.ec.europa.eu/resources-and-tools/statistics-and-factsheets/statistics/for-researchers).

The analysis will be structured across three levels of granularity:
1. **Global** – European-wide trends, how the program has developed in the past decade.
2. **Italy** – A narrower focus on italian universities: number of participants, preferred destinations, distribution across departments and the temporal evolution of the programmes.
3. **UNIMI (UMIL)** – Lastly, a deep-dive into Università degli Studi di Milano and how it has performed compared to other universities in the same city.

### Goals of this notebook
This notebook implements the **extraction and cleaning** of the data needed for the Erasmus KA1 Mobility Analysis project.
In order to prepare our data properly, we will:
- Merge 11 years of raw Erasmus KA1 datasets (2014–2024) into a single, consistent dataframe, standardizing column names in the process.
- Filter the data to retain only **university students**, both studies or traineeships.
- Remove missing values, provided they are sufficiently low in number, and irrelevant columns.
- Map the `Field of Education` labels to enable consistent analysis by academic field or department.



## 1. Data Loading and Merging

In [19]:
import pandas as pd
import alive_progress
import json

In [20]:
FILE_PATHS = [
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2024.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2023.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2022.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2021.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2020.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2019.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2018.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2017.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2016.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2015.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2014.xlsx',
]

### 1.1 Loading all yearly datasets

Each academic year is stored separately on a `.xlsx` file. As the combined filtered dataset is sufficiently small, a simple iterative load-and-concatenate approach was preferred compared to a chunked processing. Once a dataset has been loaded, its column names are normalised in order to handle inconsistencies between dataset versions (e.g. `Organisation` → `Organization`, `Mobility Duration - calendar days` → `Mobility Duration`).

As we focus exclusively on **university student exchanges**, we retain only rows where `Activity (mob)` corresponds to:
- Student mobility for studies (HE-SMS, HE-LM-SMS, between Programme Countries)
- Student mobility for traineeships (HE-SMT, HE-LM-SMT, between Programme Countries)

Everything else is discarded.

In [21]:
completeDF = pd.DataFrame()

removeCol = [
        'Mobility Start Month', 
        'Activity (mob)',
        'Field',
        'Participant Profile',
        'Mobility Start Year/Month',
        'Actual Participants',
        'Actual Participants ',
        'Project Reference',
        'Actual Participants (Contracted Projects)'
        ]

In [22]:
with alive_progress.alive_bar(len(FILE_PATHS), force_tty=True) as bar:
    for dataset in FILE_PATHS:
        df = pd.read_excel(dataset)
        #First thing, we should check the name of the columns and make them consistent across all datasets
        if 'Mobility Duration - calendar days' in df.columns:
            df.rename(columns={'Mobility Duration - calendar days': 'Mobility Duration'}, inplace=True)
        
        if 'Sending Organisation' in df.columns:
            df.rename(columns={'Sending Organisation': 'Sending Organization'}, inplace=True)

        if 'Receiving Organisation' in df.columns:
            df.rename(columns={'Receiving Organisation': 'Receiving Organization' }, inplace=True)
        
        # We are only interested in partecipants who are exchanging students in different universities, as such we will remove all other partecipants from the dataset
        temporaryDF = df[df["Activity (mob)"].isin(['HE-LM-SMS - Student mobility for studies', 
                                                   'HE-SMS - Student mobility for studies', 
                                                   'HE-SMT - Student mobility for traineeships', 
                                                   'HE-LM-SMT - Student mobility for traineeships', 
                                                   'Student mobility for studies between Programme Countries',
                                                   'Student mobility for traineeships between Programme Countries'
                                                ])]
        
        completeDF = pd.concat([completeDF, temporaryDF])
        # Save some memory
        del df, temporaryDF
        #Move the animated progress bar
        bar()

|████████████████████████████████████████| 11/11 [100%] in 6:04.4 (0.03/s)      ▁ 0/11 [0%] in 36s (~0s, 0.0/s)  ▃▁▃ 0/11 [0%] in 48s (~0s, 0.0/s) 


### 1.2 Column selection and rationale

After combining our datasets and filtering out all non university students, we can now focus on the columns of the combined dataset we obtained. Several columns are either redundant or no longer needed, and as such we can safely drop them:

| Column | Reason for removal |
|---|---|
| `Mobility Start Month` / `Mobility Start Year/Month` | Redundant with `Academic Year`. |
| `Activity (mob)` | Used in order to filter non students out, all remaining rows are students. Could still be used to evalutate exchanges / traineeships. |
| `Field` | Redundant with `Field of Education`. |
| `Participant Profile` | As all remaining rows are students, it is now useless(all `Learner`) |
| `Actual Participants`, `Actual Participants `, `Actual Participants (Contracted Projects)` | Not relevant for our analysis. |
| `Project Reference` | Not relevant for our analysis. |



After dropping the columns listed above, we drop possible remaining missing values and possible duplicates present in the dataset.

In [23]:
filteredDF = completeDF.drop(removeCol, axis=1)
filteredDF = filteredDF.dropna()
filteredDF.drop_duplicates()

,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending City,Sending Organization,Receiving Country,Receiving City,Receiving Organization
1358,2023-24,61,"0410 - Business and administration, not furthe...",Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,AT - Austria,WIEN,WU,DE - Germany,Munich,-
1359,2023-24,101,0311 - Economics,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,0,27,AT - Austria,WIEN,WU,FR - France,Toulouse,Université Toulouse Capitole
1360,2023-24,121,"0410 - Business and administration, not furthe...",Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,AT - Austria,WIEN,WU,FR - France,Paris,-
1361,2023-24,171,"0410 - Business and administration, not furthe...",Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,22,AT - Austria,WIEN,WU,FR - France,Paris,-
1365,2023-24,86,"0410 - Business and administration, not furthe...",Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,23,AT - Austria,WIEN,WU,DE - Germany,Hamburg,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215516,2014-2015,212,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,FR - France,Denain,College Bayard
215517,2014-2015,243,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,ES - Spain,Madrid,I.E.S. Isabel La Catolica
215518,2014-2015,243,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,FR - France,Alen�on,Lycee Alain
215519,2014-2015,261,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,No,18,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,ES - Spain,Mostoles,CEIP CELSO EMILIO FERREIRO


In [24]:
#Just to verify everything is working properly
print(filteredDF.isnull().sum())

Academic Year             0
Mobility Duration         0
Field of Education        0
Participant Country       0
Education Level           0
Participant Gender        0
Fewer Opportunities       0
Participant Age           0
Sending Country           0
Sending City              0
Sending Organization      0
Receiving Country         0
Receiving City            0
Receiving Organization    0
dtype: int64


## 1.3 Column Cleaning and Standardization

The field 'Field of Education' contains the subject of study of each student, but it is basically impossible to use if left as is: each university has its own naming scheme, and the same course might show slight grammatical variations or additional codes.
Thus, we must map all these label to a new set of standardized ones. Here we can see an example of such mapping: 

 `0410 - Business and administration, not further defined`,  
 `041 - Business and administration` 
 -> `Business and administration`

This is accomplished by means of a lookup table, through which we substitute each label with its canonical variant while leaving unchanged any label not found in it.

Lastly, we fix whatever other issue we might have:
N/D as a 

In [25]:
# Fix courses names
with open('mapping_results.json', 'r') as map:
    mapping_data = json.load(map)

lookup_map = mapping_data.get('flat_lookup', {})
filteredDF['Field of Education'] = filteredDF['Field of Education'].map(lookup_map).fillna(filteredDF['Field of Education'])

# Fix Uni names
with open('university_flat_mapping.json', 'r', encoding='utf-8') as uni:
    lookup_map = json.load(uni)

filteredDF['Sending Organization'] = filteredDF['Sending Organization'].map(lookup_map).fillna(filteredDF['Sending Organization'])
filteredDF['Receiving Organization'] = filteredDF['Receiving Organization'].map(lookup_map).fillna(filteredDF['Receiving Organization'])

# This to fix years
filteredDF['Academic Year'] = filteredDF['Academic Year'].str.split('-').str[0]

# This to remove codes from Contries names
filteredDF["Receiving Country"] = (filteredDF["Receiving Country"].astype(str).str.split(" - ", n=1).str[1])

filteredDF["Sending Country"] = (filteredDF["Sending Country"].astype(str).str.split(" - ", n=1).str[1])

display(filteredDF)

,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending City,Sending Organization,Receiving Country,Receiving City,Receiving Organization
1358,2023,61,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,Austria,WIEN,WU,Germany,Munich,-
1359,2023,101,Economics,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,0,27,Austria,WIEN,WU,France,Toulouse,Université Toulouse Capitole
1360,2023,121,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,Austria,WIEN,WU,France,Paris,-
1361,2023,171,Business and administration,Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,22,Austria,WIEN,WU,France,Paris,-
1365,2023,86,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,23,Austria,WIEN,WU,Germany,Hamburg,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215516,2014,212,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,France,Denain,College Bayard
215517,2014,243,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,Spain,Madrid,I.E.S. Isabel La Catolica
215518,2014,243,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,France,Alen�on,Lycee Alain
215519,2014,261,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,No,18,United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,Spain,Mostoles,CEIP CELSO EMILIO FERREIRO


## 1.4 Export

The cleaned dataframe (`filteredDF`) is exported as a CSV for use in subsequent analysis notebooks. The final dataset contains **~3.17 million rows** and **14 columns**, covering academic years 2014–2024.

In [26]:
filteredDF.to_csv('Datasets/Processed/Erasmus-Data.csv')